In [2]:
import pandas as pd

In [3]:
df_electric = pd.read_csv('all_country_data.csv')
df_weather = pd.read_csv('all_country_20.csv')

In [4]:
df_electric.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87880 entries, 0 to 87879
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   entity          87880 non-null  object 
 1   entity code     87880 non-null  object 
 2   date            87880 non-null  object 
 3   series          87880 non-null  object 
 4   generation_TWh  87880 non-null  float64
dtypes: float64(1), object(4)
memory usage: 3.4+ MB


In [5]:
df_electric.head()

,entity,entity code,date,series,generation_TWh
0,Australia,AUS,2018-01-01,Demand,19.96
1,Australia,AUS,2018-01-01,Clean,16.38
2,Australia,AUS,2018-01-01,Fossil,83.62
3,Australia,AUS,2018-01-01,Gas and Other Fossil,12.68
4,Australia,AUS,2018-01-01,"Hydro, Bioenergy and Other Renewables",5.61


In [6]:
df_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2016 entries, 0 to 2015
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           2016 non-null   object 
 1   precipitation  2016 non-null   float64
 2   solar          2016 non-null   float64
 3   humidity       2016 non-null   float64
 4   temperature    2016 non-null   float64
 5   area           2016 non-null   object 
dtypes: float64(4), object(2)
memory usage: 94.6+ KB


In [7]:
df_weather.head()

,date,precipitation,solar,humidity,temperature,area
0,01/01/2018,39.711415,41.666153,90.716614,-1.162657,Romania
1,02/01/2018,60.477952,48.278819,89.058387,-2.042290,Romania
2,03/01/2018,81.507487,84.428915,89.057320,1.208301,Romania
3,04/01/2018,25.752936,158.647809,75.713122,13.300846,Romania
4,05/01/2018,61.995416,188.878439,70.209946,17.078283,Romania


In [8]:


# 1. Chuyển đổi cột 'date' sang kiểu datetime (báo cho pandas biết gốc là MM/DD/YYYY)
df_weather['date'] = pd.to_datetime(df_weather['date'], format='%m/%d/%Y')

# 2. Chuyển ngược lại thành chuỗi theo định dạng DD/MM/YYYY
df_weather['date'] = df_weather['date'].dt.strftime('%d/%m/%Y')

df_weather['date'] = pd.to_datetime(df_weather['date'], format="%d/%m/%Y")

# Kiểm tra kết quả
print(df_weather.head())

        date  precipitation       solar   humidity  temperature     area
0 2018-01-01      39.711415   41.666153  90.716614    -1.162657  Romania
1 2018-02-01      60.477952   48.278819  89.058387    -2.042290  Romania
2 2018-03-01      81.507487   84.428915  89.057320     1.208301  Romania
3 2018-04-01      25.752936  158.647809  75.713122    13.300846  Romania
4 2018-05-01      61.995416  188.878439  70.209946    17.078283  Romania


In [9]:
# Trước tiên, đảm bảo cột 'date' của df_electric có kiểu datetime giống với df_weather
df_electric['date'] = pd.to_datetime(df_electric['date'])

# Thực hiện merge 2 dataframe
# Sử dụng how='left' nhằm đảm bảo giữ nguyên toàn bộ số dòng của bảng điện lưới (df_electric)
df_merged = pd.merge(
    df_electric,
    df_weather,
    left_on=['entity', 'date'],
    right_on=['area', 'date'],
    how='left'
)

# (Tuỳ chọn) Cột 'area' sau khi merge sẽ bị thừa (do nó giống 'entity'), nên có thể xóa đi
if 'area' in df_merged.columns:
    df_merged = df_merged.drop(columns=['area'])

# Kiểm tra thử kết quả
df_merged.head()


,entity,entity code,date,series,generation_TWh,precipitation,solar,humidity,temperature
0,Australia,AUS,2018-01-01,Demand,19.96,91.269472,222.716523,45.105328,29.743529
1,Australia,AUS,2018-01-01,Clean,16.38,91.269472,222.716523,45.105328,29.743529
2,Australia,AUS,2018-01-01,Fossil,83.62,91.269472,222.716523,45.105328,29.743529
3,Australia,AUS,2018-01-01,Gas and Other Fossil,12.68,91.269472,222.716523,45.105328,29.743529
4,Australia,AUS,2018-01-01,"Hydro, Bioenergy and Other Renewables",5.61,91.269472,222.716523,45.105328,29.743529


In [10]:
# Đảm bảo cột date có cùng định dạng
df_electric['date'] = pd.to_datetime(df_electric['date'])

# Chỉnh lại tên quốc gia bị lệch trong df_electric cho khớp với df_weather
df_electric['entity'] = df_electric['entity'].replace({'Philippines (the)': 'Philippines'})

# Lọc bỏ quốc gia 'Morocco' ra khỏi df_electric (vì df_weather không có dữ liệu để ghép)
df_electric = df_electric[df_electric['entity'] != 'Morocco']

# Thực hiện merge 2 dataframe
df_merged = pd.merge(
    df_electric,
    df_weather,
    left_on=['entity', 'date'],
    right_on=['area', 'date'],
    how='left'
)

# Cột 'area' sau khi merge sẽ bị thừa (vì giống 'entity'), nên xóa để gọn dữ liệu
if 'area' in df_merged.columns:
    df_merged = df_merged.drop(columns=['area'])

# Kiểm tra lại kết quả tổng quát
df_merged.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84190 entries, 0 to 84189
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   entity          84190 non-null  object        
 1   entity code     84190 non-null  object        
 2   date            84190 non-null  datetime64[ns]
 3   series          84190 non-null  object        
 4   generation_TWh  84190 non-null  float64       
 5   precipitation   84190 non-null  float64       
 6   solar           84190 non-null  float64       
 7   humidity        84190 non-null  float64       
 8   temperature     84190 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(3)
memory usage: 5.8+ MB


In [11]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84190 entries, 0 to 84189
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   entity          84190 non-null  object        
 1   entity code     84190 non-null  object        
 2   date            84190 non-null  datetime64[ns]
 3   series          84190 non-null  object        
 4   generation_TWh  84190 non-null  float64       
 5   precipitation   84190 non-null  float64       
 6   solar           84190 non-null  float64       
 7   humidity        84190 non-null  float64       
 8   temperature     84190 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(3)
memory usage: 5.8+ MB


In [12]:
df_electric['entity'].unique()

array(['Australia', 'Austria', 'Chile', 'Colombia', 'Czechia', 'France',
       'Germany', 'Kazakhstan', 'Malaysia', 'Pakistan', 'Philippines',
       'Poland', 'Serbia', 'South Africa', 'Spain', 'Sweden', 'Taiwan',
       'Turkey', 'Ukraine', 'Viet Nam'], dtype=object)

In [13]:
# Lưu dữ liệu ra một file CSV mới
df_merged.to_csv('merged_electric_weather_data.csv', index=False)

print("Đã lưu dữ liệu thành công!")


Đã lưu dữ liệu thành công!
